# PYNQ scalar demo
先核准同板制品、运行环境与 PL 独占权，并取得本轮下载/运行授权。默认只检查文件。Python 包需已部署到目标环境；不在此 Notebook 自动安装。


In [ ]:
from pathlib import Path
from pynq_scalar_demo.bundle import validate_bundle
HARDWARE_AUTHORIZED = False
BUNDLE = Path('/home/xilinx/bundle-001')
manifest = validate_bundle(BUNDLE)
print(manifest['scope'], manifest['build_id'])


加载单元格可重复执行，同一 manifest 返回原所有者。unknown 状态下不能靠重复加载恢复；不得由其他代码下载 PL。


In [ ]:
if HARDWARE_AUTHORIZED:
    from pynq_scalar_demo.board import open_board
    ip = open_board(BUNDLE / 'scalar.bit', manifest['ip_name'], manifest['file_identity'])
    print('state:', ip.session.state)
else:
    print('hardware NOT_RUN')


In [ ]:
if HARDWARE_AUTHORIZED:
    for x, bias in ((0, 0), (3, 4), (0xffffffff, 1), (0x80000000, 0x80000000)):
        actual = ip.add(x, bias)
        assert actual == ((x + bias) & 0xffffffff)
        print(x, bias, actual)


IRQ 验收必须收到真实中断。Jupyter 用顶层 await；超时/取消后中断对象不能直接复用，按 references/interrupts.md 处理。


In [ ]:
if HARDWARE_AUTHORIZED:
    before = ip.session.irq_events
    actual = await ip.add_async(0xffffffff, 1, timeout=1.0)
    assert actual == 0 and ip.session.irq_events == before + 1
    print('actual IRQ events:', ip.session.irq_events)


In [ ]:
if HARDWARE_AUTHORIZED:
    ip.session.close()
    print('end state:', ip.session.state)


记录目标板、工具/PYNQ/内核版本、manifest 身份、结果和实际操作。该模板没有执行输出；四向量通过不能代替取消、重入和物理冷启动验收。
